<a href="https://colab.research.google.com/github/sami01979/ai_text_detection/blob/main/recobering_deberta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
R = "/content/drive/MyDrive/"

def check_model(path):
    if not os.path.isdir(path):
        print("MISSING FOLDER:", path); return
    files = os.listdir(path)
    miss = [f for f in ["config.json", "model.safetensors", "tokenizer_config.json"] if f not in files]
    tok_ok = any(f in files for f in ["tokenizer.json", "spm.model", "vocab.txt"])
    mb = round(os.path.getsize(path + "/model.safetensors") / 1e6) if "model.safetensors" in files else None
    print("OK     " if not miss and tok_ok else "PROBLEM", path, "| missing:", miss, "| tokenizer files:", tok_ok, "| model MB:", mb)

check_model(R + "final_model_v1")
check_model(R + "thesis_run_deberta/final_model_deberta_v1")
check_model(R + "thesis_run_electra/final_model_electra_mage")

print()
D = R + "thesis_data/"
print("thesis_data:", sorted(os.listdir(D)) if os.path.isdir(D) else "MISSING FOLDER")
for f, expect in [("val.csv", 9951), ("test.csv", 9952), ("mage_train.csv", 120000),
                  ("mage_val.csv", 15000), ("mage_test.csv", 15000)]:
    p = D + f
    n = len(pd.read_csv(p, usecols=["label"])) if os.path.exists(p) else None
    print(f"{f:16s}", f"{n} rows (expect {expect})" if n else "MISSING")

print()
for d in ["thesis_run_electra", "thesis_run_deberta"]:
    p = R + d
    print(d, "->", sorted(os.listdir(p)) if os.path.isdir(p) else "MISSING")
!df -h /content/drive | tail -1

Mounted at /content/drive
OK      /content/drive/MyDrive/final_model_v1 | missing: [] | tokenizer files: True | model MB: 499
PROBLEM /content/drive/MyDrive/thesis_run_deberta/final_model_deberta_v1 | missing: ['model.safetensors'] | tokenizer files: True | model MB: None
MISSING FOLDER: /content/drive/MyDrive/thesis_run_electra/final_model_electra_mage

thesis_data: ['mage_test.csv', 'mage_train.csv', 'mage_val.csv', 'test.csv', 'train_xl.csv', 'val.csv']
val.csv          9951 rows (expect 9951)
test.csv         9952 rows (expect 9952)
mage_train.csv   120000 rows (expect 120000)
mage_val.csv     15000 rows (expect 15000)
mage_test.csv    15000 rows (expect 15000)

thesis_run_electra -> ['checkpoint-2000', 'checkpoint-3000']
thesis_run_deberta -> ['final_model_deberta_v1']
drive            15G  7.9G  7.2G  53% /content/drive


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for f in files:
        if f.endswith(".safetensors"):
            p = os.path.join(root, f)
            print(f"{round(os.path.getsize(p) / 1e6):5d} MB  {p}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  499 MB  /content/drive/MyDrive/final_model/model.safetensors
  499 MB  /content/drive/MyDrive/final_model_v1/model.safetensors
  738 MB  /content/drive/MyDrive/thesis_run_deberta/checkpoint-6000/model.safetensors
  438 MB  /content/drive/MyDrive/thesis_run_electra/checkpoint-2000/model.safetensors
  438 MB  /content/drive/MyDrive/thesis_run_electra/checkpoint-3000/model.safetensors


In [ ]:
import os, shutil, torch
from google.colab import drive
drive.mount('/content/drive')
from transformers import AutoTokenizer, AutoModelForSequenceClassification

base = "/content/drive/MyDrive/thesis_run_deberta"
src, dst = base + "/checkpoint-6000", base + "/final_model_deberta_v1"

shutil.copy(src + "/model.safetensors", dst + "/model.safetensors")
print("final folder:", os.listdir(dst), "|", round(os.path.getsize(dst + "/model.safetensors") / 1e6), "MB")

tok = AutoTokenizer.from_pretrained(dst)
model = AutoModelForSequenceClassification.from_pretrained(dst).float().eval()
text = ("Climate change is one of the most pressing challenges facing humanity today. Rising global "
        "temperatures contribute to more frequent extreme weather events, melting ice caps, and rising "
        "sea levels. Addressing this issue requires a multifaceted approach, including transitioning to "
        "renewable energy sources, improving energy efficiency, and fostering international cooperation.")
enc = tok(text, truncation=True, max_length=256, return_tensors="pt")
with torch.no_grad():
    print("AI probability on a known AI paragraph:", torch.softmax(model(**enc).logits, dim=-1)[0][1].item())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
final folder: ['config.json', 'tokenizer_config.json', 'tokenizer.json', 'model.safetensors'] | 738 MB


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

AI probability on a known AI paragraph: 0.9999805688858032


In [1]:
!pip install -q transformers datasets accelerate

In [2]:
import numpy as np, pandas as pd, torch
from google.colab import drive
drive.mount('/content/drive')
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

assert torch.cuda.is_available(), "NO GPU - set the runtime to T4 first"
D = "/content/drive/MyDrive/thesis_data/"
path = "/content/drive/MyDrive/thesis_run_deberta/final_model_deberta_v1"
val, test = pd.read_csv(D + "val.csv"), pd.read_csv(D + "test.csv")

tok = AutoTokenizer.from_pretrained(path)
model = AutoModelForSequenceClassification.from_pretrained(path).float().cuda().eval()

def probs(texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], truncation=True, max_length=256, padding=True, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out.append(torch.softmax(model(**enc).logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(out)

pv, pt = probs(val.text.tolist()), probs(test.text.tolist())
yv, yt = val.label.values, test.label.values
np.savez("/content/drive/MyDrive/deberta_probs_ckpt6000.npz", pv=pv, pt=pt, yv=yv, yt=yt)

ths = np.concatenate([np.arange(0.05, 0.95, 0.01), np.arange(0.95, 0.9995, 0.001)])
accs = [accuracy_score(yv, pv >= t) for t in ths]
best_t = ths[int(np.argmax(accs))]
pred = (pt >= best_t).astype(int)
print(f"best threshold (VALIDATION): {best_t:.3f} | val acc {max(accs):.4f}")
print(f"TEST acc @0.5 : {accuracy_score(yt, pt >= 0.5):.4f}")
print(f"TEST acc @tuned: {accuracy_score(yt, pred):.4f} | ROC-AUC {roc_auc_score(yt, pt):.4f}")
print(confusion_matrix(yt, pred))
print(classification_report(yt, pred, target_names=["Human", "AI"], digits=4))

Mounted at /content/drive


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

best threshold (VALIDATION): 0.997 | val acc 0.9633
TEST acc @0.5 : 0.9425
TEST acc @tuned: 0.9642 | ROC-AUC 0.9940
[[4811  165]
 [ 191 4785]]
              precision    recall  f1-score   support

       Human     0.9618    0.9668    0.9643      4976
          AI     0.9667    0.9616    0.9641      4976

    accuracy                         0.9642      9952
   macro avg     0.9642    0.9642    0.9642      9952
weighted avg     0.9642    0.9642    0.9642      9952

